# Analyse des Correspondances Multiples (ACM)
## Dataset: Loisirs

Notebook interactif pour reproduire l'analyse MCA du script R avec visualisations en Python.

## 1. Installation des dépendances

In [ ]:
import sys
import subprocess

# Installer les packages requis
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'prince', 'scikit-learn', 'adjustText']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("✓ All dependencies installed!")

## 2. Imports et configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse
import prince

try:
    from adjustText import adjust_text
    HAS_ADJUSTTEXT = True
except:
    HAS_ADJUSTTEXT = False
    print("⚠ adjustText not available - labels may overlap")

# Configuration matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful!")

## 3. Chargement des données

In [ ]:
# Le fichier CSV doit être dans le même dossier que ce notebook
import os
CSV_PATH = os.path.join(os.getcwd(), 'AnaDo_JeuDonnees_Loisirs.csv')

# Vérifier si le fichier existe
if not os.path.exists(CSV_PATH):
    print(f"❌ ERREUR: Fichier CSV non trouvé à {CSV_PATH}")
    print("Solution: Place le fichier dans le même dossier que le notebook.")
    raise FileNotFoundError(f"CSV file not found at {CSV_PATH}")

print(f"✓ CSV trouvé: {CSV_PATH}")

# Charger le dataset
df = pd.read_csv(CSV_PATH, sep=';', header=0, encoding='utf-8', low_memory=False)
print(f"\n✓ Dataset chargé: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\nAperçu:")
print(df.head())
print(f"\nTypes de données:")
print(df.dtypes)

## 4. Prétraitement des données

In [ ]:
# Convertir les colonnes object en catégories
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].astype('category')

print(f"✓ Colonnes converties en catégories: {df.select_dtypes(include=['category']).shape[1]}")
print(f"\nColonnes catégoriques: {df.select_dtypes(include=['category']).columns.tolist()}")

# Vérifier les données manquantes
missing = df.isnull().sum()
if missing.sum() > 0:
    print(f"\n⚠ Valeurs manquantes:")
    print(missing[missing > 0])
else:
    print(f"\n✓ Pas de valeurs manquantes")

## 5. Configuration des variables supplémentaires

**IMPORTANT**: Modifiez les indices ci-dessous selon votre dataset:
- R utilisait `quali.sup = 19:22` (indices 1-based) et `quanti.sup = 23`

In [ ]:
# PARAMÈTRES À AJUSTER SELON VOTRE DATASET
QUALI_SUP_INDICES = [19, 20, 21, 22]  # Indices 1-based
QUANTI_SUP_INDEX = 23                  # Indice 1-based

# Convertir en indices 0-based
quali_sup_cols = []
for idx in QUALI_SUP_INDICES:
    if 1 <= idx <= df.shape[1]:
        quali_sup_cols.append(df.columns[idx - 1])

quanti_sup_col = None
if 1 <= QUANTI_SUP_INDEX <= df.shape[1]:
    quanti_sup_col = df.columns[QUANTI_SUP_INDEX - 1]

print(f"✓ Variables qualitatives supplémentaires: {quali_sup_cols}")
print(f"✓ Variable quantitative supplémentaire: {quanti_sup_col}")

# Colonnes pour l'ACM (exclure quali.sup)
cat_cols = df.select_dtypes(include=['category']).columns.tolist()
mca_cols = [c for c in cat_cols if c not in quali_sup_cols]

print(f"\n✓ Colonnes utilisées pour l'ACM ({len(mca_cols)}):")
print(mca_cols)

## 6. Réalisation de l'ACM (MCA)

In [ ]:
# Préparer les données
X = df[mca_cols].copy()
X = X.apply(lambda col: col.astype(str))

# Fit MCA
print("⏳ Calcul de l'ACM... (quelques secondes)")
mca = prince.MCA(n_components=5, n_iter=10, copy=True, random_state=42)
mca = mca.fit(X)

print("✓ ACM terminée!")

# Récupérer les résultats
eig = mca.eigenvalues_
explained = mca.explained_inertia_
row_coords = mca.row_coordinates(X)
col_coords = mca.column_coordinates(X)

print(f"\n✓ Inertie expliquée par dimension:")
for i, exp in enumerate(explained):
    print(f"  Dim {i+1}: {exp*100:.2f}%")

## 7. Graphique des inerties (Eigenvalues)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(1, len(explained) + 1), np.array(explained) * 100)
ax.set_xlabel('Dimension', fontsize=12)
ax.set_ylabel('Inertie expliquée (%)', fontsize=12)
ax.set_title('Eigenvalues / Inertie expliquée', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total inertie expliquée (5 dims): {sum(explained)*100:.2f}%")

## 8. Graphique des individus (axes 1 & 2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
hue_col = quali_sup_cols[0] if len(quali_sup_cols) > 0 else None

if hue_col:
    hue_values = df[hue_col].astype(str)
    palette = sns.color_palette('tab10', n_colors=len(hue_values.unique()))
    sns.scatterplot(x=row_coords[0], y=row_coords[1], hue=hue_values, 
                    palette=palette, s=50, ax=ax, alpha=0.7, legend='full')
    ax.legend(title=hue_col, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
else:
    ax.scatter(row_coords[0], row_coords[1], s=30, alpha=0.7)

ax.axhline(0, color='grey', lw=0.7, linestyle='--')
ax.axvline(0, color='grey', lw=0.7, linestyle='--')
ax.set_xlabel(f'Dimension 1 ({explained[0]*100:.2f}%)', fontsize=12)
ax.set_ylabel(f'Dimension 2 ({explained[1]*100:.2f}%)', fontsize=12)
ax.set_title('Individus (axes 1 & 2)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Graphique des individus (axes 2 & 3)

In [ ]:
if row_coords.shape[1] >= 3:
    fig, ax = plt.subplots(figsize=(10, 8))
    if hue_col:
        sns.scatterplot(x=row_coords[1], y=row_coords[2], hue=df[hue_col].astype(str), 
                        s=50, ax=ax, alpha=0.7, palette='tab10')
        ax.legend(title=hue_col, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
    else:
        ax.scatter(row_coords[1], row_coords[2], s=30, alpha=0.7)
    
    ax.axhline(0, color='grey', lw=0.7, linestyle='--')
    ax.axvline(0, color='grey', lw=0.7, linestyle='--')
    ax.set_xlabel(f'Dimension 2 ({explained[1]*100:.2f}%)', fontsize=12)
    ax.set_ylabel(f'Dimension 3 ({explained[2]*100:.2f}%)', fontsize=12)
    ax.set_title('Individus (axes 2 & 3)', fontsize=14, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠ Pas assez de dimensions pour axes 2 & 3")

## 10. Graphique des catégories (modalités)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

xs = col_coords[0]
ys = col_coords[1]

ax.scatter(xs, ys, s=50, alpha=0.7)

# Ajouter labels
texts = []
for i, label in enumerate(col_coords.index):
    texts.append(ax.text(xs[i], ys[i], label, fontsize=9, ha='center'))

if HAS_ADJUSTTEXT:
    adjust_text(texts, arrowprops=dict(arrowstyle='->', color='grey', alpha=0.3))

ax.axhline(0, color='grey', lw=0.7, linestyle='--')
ax.axvline(0, color='grey', lw=0.7, linestyle='--')
ax.set_xlabel(f'Dimension 1 ({explained[0]*100:.2f}%)', fontsize=12)
ax.set_ylabel(f'Dimension 2 ({explained[1]*100:.2f}%)', fontsize=12)
ax.set_title('Catégories des variables', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Nombre de catégories: {len(col_coords)}")

## 11. Ellipses de confiance par modalité

In [ ]:
def draw_ellipse(ax, x, y, n_std=1.96, **kwargs):
    if len(x) < 2:
        return None
    cov = np.cov(np.vstack([x, y]))
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=(np.mean(x), np.mean(y)), width=width, height=height, 
                   angle=theta, **kwargs)
    ax.add_patch(ell)
    return ell

group_var = hue_col if hue_col else (mca_cols[0] if len(mca_cols) > 0 else None)

if group_var:
    fig, ax = plt.subplots(figsize=(10, 8))
    groups = df[group_var].astype(str)
    unique = groups.unique()
    colors = sns.color_palette('tab10', n_colors=len(unique))
    
    for k, u in enumerate(unique):
        mask = groups == u
        x = row_coords.loc[mask, 0]
        y = row_coords.loc[mask, 1]
        ax.scatter(x, y, s=30, color=colors[k], alpha=0.6, label=str(u))
        
        if len(x) > 2:
            draw_ellipse(ax, x.values, y.values, n_std=1.96, 
                         edgecolor=colors[k], facecolor='none', lw=2)
    
    ax.legend(title=group_var, bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.axhline(0, color='grey', lw=0.7, linestyle='--')
    ax.axvline(0, color='grey', lw=0.7, linestyle='--')
    ax.set_xlabel(f'Dimension 1 ({explained[0]*100:.2f}%)', fontsize=12)
    ax.set_ylabel(f'Dimension 2 ({explained[1]*100:.2f}%)', fontsize=12)
    ax.set_title(f'Ellipses par modalité ({group_var})', fontsize=14, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠ Pas de variable de groupement")

## 12. Corrélations variable quantitative suppl.

In [ ]:
if quanti_sup_col is not None:
    qs = pd.to_numeric(df[quanti_sup_col], errors='coerce')
    
    corrs = []
    for dim in range(min(5, row_coords.shape[1])):
        corr = np.corrcoef(qs.fillna(qs.mean()), row_coords[dim])[0, 1]
        corrs.append(corr)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    dims = list(range(1, len(corrs) + 1))
    ax.bar(dims, np.abs(corrs), color='steelblue', alpha=0.7)
    ax.set_xlabel('Dimension', fontsize=12)
    ax.set_ylabel('|Corrélation|', fontsize=12)
    ax.set_title(f'Corrélations: {quanti_sup_col}', fontsize=14, fontweight='bold')
    ax.set_xticks(dims)
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Corrélations avec {quanti_sup_col}:")
    for i, c in enumerate(corrs):
        print(f"  Dim {i+1}: {c:.4f}")
else:
    print("⚠ Pas de variable quantitative supplémentaire")

## 13. Résumé statistique

In [ ]:
print("=" * 60)
print("RÉSUMÉ DE L'ANALYSE MCA")
print("=" * 60)
print(f"\nDataset: {df.shape[0]} individus × {df.shape[1]} variables")
print(f"Colonnes pour MCA: {len(mca_cols)}")
print(f"Nombre de catégories: {len(col_coords)}")
print(f"\nVariables quali.sup: {quali_sup_cols if quali_sup_cols else 'Aucune'}")
print(f"Variable quanti.sup: {quanti_sup_col if quanti_sup_col else 'Aucune'}")
print(f"\nInertie cumulée:")
cum_exp = 0
for i, exp in enumerate(explained[:5]):
    cum_exp += exp
    print(f"  Dim {i+1}: {cum_exp*100:.2f}%")
print("\n" + "=" * 60)

## 14. Exporter les résultats

In [ ]:
out_dir = 'mca_results'
os.makedirs(out_dir, exist_ok=True)

# Sauvegarder les coordonnées
row_coords.to_csv(os.path.join(out_dir, 'row_coordinates.csv'))
col_coords.to_csv(os.path.join(out_dir, 'column_coordinates.csv'))

# Sauvegarder les inerties
inertia_df = pd.DataFrame({
    'Dimension': range(1, len(explained) + 1),
    'Eigenvalue': eig,
    'Explained_Inertia': explained,
    'Cumulative_Inertia': np.cumsum(explained)
})
inertia_df.to_csv(os.path.join(out_dir, 'inertia.csv'), index=False)

print(f"✓ Résultats exportés dans '{out_dir}/':")
print(f"  - row_coordinates.csv")
print(f"  - column_coordinates.csv")
print(f"  - inertia.csv")

# Analyse des Correspondances Multiples (ACM)
## Dataset: Loisirs

Notebook interactif pour reproduire l'analyse MCA du script R avec visualisations en Python.

## 1. Installation des dépendances

In [7]:
import sys
import subprocess

# Installer les packages requis
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'prince', 'scikit-learn', 'adjustText']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("All dependencies installed!")

Installing scikit-learn...
All dependencies installed!


## 2. Imports et configuration

In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse
import prince

try:
    from adjustText import adjust_text
    HAS_ADJUSTTEXT = True
except:
    HAS_ADJUSTTEXT = False
    print("adjustText not available - labels may overlap")

# Configuration matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All imports successful!")

All imports successful!


## 3. Chargement des données

In [9]:
# MODIFIER CE CHEMIN si nécessaire
# Le fichier se trouve dans le même dossier que le notebook
import os
CSV_PATH = os.path.join(os.getcwd(), 'AnaDo_JeuDonnees_Loisirs.csv')
print(f"Recherche du CSV: {CSV_PATH}")

# Charger le dataset
df = pd.read_csv(CSV_PATH, sep=';', header=0, encoding='utf-8', low_memory=False)
print(f"Dataset chargé: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\nAperçu:")

<VSCode.Cell id="#VSC-815174c5" language="markdown">
print(df.head())print(df.info())

## 4. Prétraitement des donnéesprint(f"\nInfo:")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\khadi\\Downloads\\ACM\\ACM\\EXEMPLE DE COURS\\AnaDo_JeuDonnees_Loisirs.csv'

In [ ]:
# Convertir les colonnes object en catégories
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].astype('category')

print("Colonnes converties en catégories:")
print(df.select_dtypes(include=['category']).columns.tolist())

# Vérifier les données manquantes
print(f"\nValeurs manquantes:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

## 5. Configuration des variables supplémentaires

**IMPORTANT**: Modifiez les indices ci-dessous selon votre dataset:
- R utilisait `quali.sup = 19:22` (indices 1-based) et `quanti.sup = 23`
- Les indices ci-dessous sont 1-based (ajustez-les si les colonnes sont différentes)

In [ ]:
# PARAMÈTRES À AJUSTER SELON VOTRE DATASET
QUALI_SUP_INDICES = [19, 20, 21, 22]  # Indices 1-based pour quali.sup
QUANTI_SUP_INDEX = 23                  # Indice 1-based pour quanti.sup

# Convertir en indices 0-based et récupérer les noms de colonnes
quali_sup_cols = []
for idx in QUALI_SUP_INDICES:
    if 1 <= idx <= df.shape[1]:
        quali_sup_cols.append(df.columns[idx - 1])

quanti_sup_col = None
if 1 <= QUANTI_SUP_INDEX <= df.shape[1]:
    quanti_sup_col = df.columns[QUANTI_SUP_INDEX - 1]

print(f"Variables qualitatives supplémentaires: {quali_sup_cols}")
print(f"Variable quantitative supplémentaire: {quanti_sup_col}")

# Colonnes à utiliser pour l'ACM (exclure quali.sup)
cat_cols = df.select_dtypes(include=['category']).columns.tolist()
mca_cols = [c for c in cat_cols if c not in quali_sup_cols]

print(f"\nColonnes utilisées pour l'ACM ({len(mca_cols)}):")
print(mca_cols)

## 6. Réalisation de l'ACM (MCA)

In [ ]:
# Préparer les données pour prince
X = df[mca_cols].copy()
X = X.apply(lambda col: col.astype(str))

# Fit MCA
print("Calcul de l'ACM... (cela peut prendre quelques secondes)")
mca = prince.MCA(n_components=5, n_iter=10, copy=True, random_state=42)
mca = mca.fit(X)

print("ACM terminée!")

# Récupérer les inerties et coordonnées
eig = mca.eigenvalues_
explained = mca.explained_inertia_
row_coords = mca.row_coordinates(X)
col_coords = mca.column_coordinates(X)

print(f"\nInertie expliquée par dimension:")
for i, (e, exp) in enumerate(zip(eig, explained)):
    print(f"  Dim {i+1}: {exp*100:.2f}%")

## 7. Graphique des inerties (Eigenvalues)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(1, len(explained) + 1), np.array(explained) * 100)
ax.set_xlabel('Dimension', fontsize=12)
ax.set_ylabel('Inertie expliquée (%)', fontsize=12)
ax.set_title('Eigenvalues / Inertie expliquée', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total inertie expliquée (5 dims): {sum(explained)*100:.2f}%")

## 8. Graphique des individus (axes 1 & 2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
hue_col = quali_sup_cols[0] if len(quali_sup_cols) > 0 else None

if hue_col:
    hue_values = df[hue_col].astype(str)
    palette = sns.color_palette('tab10', n_colors=len(hue_values.unique()))
    sns.scatterplot(x=row_coords[0], y=row_coords[1], hue=hue_values, 
                    palette=palette, s=50, ax=ax, alpha=0.7, legend='full')
    ax.legend(title=hue_col, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
else:
    ax.scatter(row_coords[0], row_coords[1], s=30, alpha=0.7)

ax.axhline(0, color='grey', lw=0.7, linestyle='--')
ax.axvline(0, color='grey', lw=0.7, linestyle='--')
ax.set_xlabel(f'Dimension 1 ({explained[0]*100:.2f}%)', fontsize=12)
ax.set_ylabel(f'Dimension 2 ({explained[1]*100:.2f}%)', fontsize=12)
ax.set_title('Individus (axes 1 & 2)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Graphique des individus (axes 2 & 3)

In [ ]:
if row_coords.shape[1] >= 3:
    fig, ax = plt.subplots(figsize=(10, 8))
    if hue_col:
        sns.scatterplot(x=row_coords[1], y=row_coords[2], hue=df[hue_col].astype(str), 
                        s=50, ax=ax, alpha=0.7, palette='tab10')
        ax.legend(title=hue_col, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
    else:
        ax.scatter(row_coords[1], row_coords[2], s=30, alpha=0.7)
    
    ax.axhline(0, color='grey', lw=0.7, linestyle='--')
    ax.axvline(0, color='grey', lw=0.7, linestyle='--')
    ax.set_xlabel(f'Dimension 2 ({explained[1]*100:.2f}%)', fontsize=12)
    ax.set_ylabel(f'Dimension 3 ({explained[2]*100:.2f}%)', fontsize=12)
    ax.set_title('Individus (axes 2 & 3)', fontsize=14, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Pas assez de dimensions pour axes 2 & 3")

## 10. Graphique des catégories (modalités des variables)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

xs = col_coords[0]
ys = col_coords[1]

ax.scatter(xs, ys, s=50, alpha=0.7)

# Ajouter les labels
texts = []
for i, label in enumerate(col_coords.index):
    texts.append(ax.text(xs[i], ys[i], label, fontsize=9, ha='center'))

# Utiliser adjustText si disponible
if HAS_ADJUSTTEXT:
    adjust_text(texts, arrowprops=dict(arrowstyle='->', color='grey', alpha=0.3))

ax.axhline(0, color='grey', lw=0.7, linestyle='--')
ax.axvline(0, color='grey', lw=0.7, linestyle='--')
ax.set_xlabel(f'Dimension 1 ({explained[0]*100:.2f}%)', fontsize=12)
ax.set_ylabel(f'Dimension 2 ({explained[1]*100:.2f}%)', fontsize=12)
ax.set_title('Catégories des variables (plan factoriel)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Nombre de catégories: {len(col_coords)}")

## 11. Ellipses de confiance par modalité

In [ ]:
def draw_ellipse(ax, x, y, n_std=1.96, **kwargs):
    """Tracer une ellipse de confiance"""
    if len(x) < 2:
        return None
    cov = np.cov(np.vstack([x, y]))
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=(np.mean(x), np.mean(y)), width=width, height=height, 
                   angle=theta, **kwargs)
    ax.add_patch(ell)
    return ell

# Tracer les ellipses
group_var = hue_col if hue_col else (mca_cols[0] if len(mca_cols) > 0 else None)

if group_var:
    fig, ax = plt.subplots(figsize=(10, 8))
    groups = df[group_var].astype(str)
    unique = groups.unique()
    colors = sns.color_palette('tab10', n_colors=len(unique))
    
    for k, u in enumerate(unique):
        mask = groups == u
        x = row_coords.loc[mask, 0]
        y = row_coords.loc[mask, 1]
        ax.scatter(x, y, s=30, color=colors[k], alpha=0.6, label=str(u))
        
        if len(x) > 2:
            draw_ellipse(ax, x.values, y.values, n_std=1.96, 
                         edgecolor=colors[k], facecolor='none', lw=2)
    
    ax.legend(title=group_var, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
    ax.axhline(0, color='grey', lw=0.7, linestyle='--')
    ax.axvline(0, color='grey', lw=0.7, linestyle='--')
    ax.set_xlabel(f'Dimension 1 ({explained[0]*100:.2f}%)', fontsize=12)
    ax.set_ylabel(f'Dimension 2 ({explained[1]*100:.2f}%)', fontsize=12)
    ax.set_title(f'Individus avec ellipses par modalité ({group_var})', fontsize=14, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Pas de variable de groupement disponible")

## 12. Corrélations avec la variable quantitative supplémentaire

In [ ]:
if quanti_sup_col is not None:
    # Convertir en numérique
    qs = pd.to_numeric(df[quanti_sup_col], errors='coerce')
    
    # Calculer les corrélations
    corrs = []
    for dim in range(min(5, row_coords.shape[1])):
        corr = np.corrcoef(qs.fillna(qs.mean()), row_coords[dim])[0, 1]
        corrs.append(corr)
    
    # Graphique
    fig, ax = plt.subplots(figsize=(8, 5))
    dims = list(range(1, len(corrs) + 1))
    ax.bar(dims, np.abs(corrs), color='steelblue', alpha=0.7)
    ax.set_xlabel('Dimension', fontsize=12)
    ax.set_ylabel('|Corrélation|', fontsize=12)
    ax.set_title(f'Corrélations avec la variable quantitative suppl.: {quanti_sup_col}', 
                 fontsize=14, fontweight='bold')
    ax.set_xticks(dims)
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print(f"\nCorrélations avec {quanti_sup_col}:")
    for i, c in enumerate(corrs):
        print(f"  Dim {i+1}: {c:.4f}")
else:
    print("Pas de variable quantitative supplémentaire")

## 13. Résumé statistique de l'ACM

In [ ]:
print("=" * 60)
print("RÉSUMÉ DE L'ANALYSE MCA")
print("=" * 60)
print(f"\nDimensions du dataset: {df.shape[0]} individus × {df.shape[1]} variables")
print(f"Colonnes utilisées pour l'ACM: {len(mca_cols)}")
print(f"Nombre de catégories: {len(col_coords)}")
print(f"\nVariables qualitatives supplémentaires: {quali_sup_cols if quali_sup_cols else 'Aucune'}")
print(f"Variable quantitative supplémentaire: {quanti_sup_col if quanti_sup_col else 'Aucune'}")
print(f"\nInertie expliquée cumulée:")
cum_exp = 0
for i, exp in enumerate(explained[:5]):
    cum_exp += exp
    print(f"  Dimension {i+1}: {cum_exp*100:.2f}%")
print("\n" + "=" * 60)

## 14. Exporter les résultats (optionnel)

Décommentez la cellule ci-dessous pour sauvegarder les coordonnées et les graphes.

In [ ]:
# Créer dossier de sortie
import os
out_dir = 'mca_results'
os.makedirs(out_dir, exist_ok=True)

# Sauvegarder les coordonnées en CSV
row_coords.to_csv(os.path.join(out_dir, 'row_coordinates.csv'))
col_coords.to_csv(os.path.join(out_dir, 'column_coordinates.csv'))

# Sauvegarder les inerties
inertia_df = pd.DataFrame({
    'Dimension': range(1, len(explained) + 1),
    'Eigenvalue': eig,
    'Explained_Inertia': explained,
    'Cumulative_Inertia': np.cumsum(explained)
})
inertia_df.to_csv(os.path.join(out_dir, 'inertia.csv'), index=False)

print(f"Résultats exportés dans le dossier '{out_dir}':")
print(f"  - row_coordinates.csv (coordonnées des individus)")
print(f"  - column_coordinates.csv (coordonnées des catégories)")
print(f"  - inertia.csv (inerties par dimension)")

## Notes et interprétation

### Paramètres à ajuster:
- **QUALI_SUP_INDICES**: Indices 1-based des colonnes qualitatives supplémentaires (par défaut: 19, 20, 21, 22)
- **QUANTI_SUP_INDEX**: Indice 1-based de la colonne quantitative supplémentaire (par défaut: 23)

### Interprétation des graphes:
1. **Eigenvalues**: Montre l'inertie expliquée par chaque dimension. Les premières dimensions capturent le plus de variance.
2. **Individus (axes 1 & 2)**: Positionne chaque individu dans l'espace factoriel.
3. **Catégories**: Montre la position des modalités (catégories) des variables.
4. **Ellipses**: Les ellipses de confiance regroupent les individus par modalité.
5. **Corrélations quanti.sup**: Mesure l'association entre la variable quantitative et les axes.

### Package prince:
Le package `prince` implémente la MCA (Multiple Correspondence Analysis), l'équivalent Python de `FactoMineR::MCA()` en R.